# BIL545 · Proje 2 · Beijing PM2.5 kirlilik epizotları

**Grup adı:** Grup 03
**Üyeler:** Süleyman Balsak (254312022) · Seda Erten (254312008) · İlkin Urcan (244329078) · Aslıhan Semiz Varlı (254329024)
**İletişim sorumlusu:** Süleyman Balsak
**Konu:** 9.7 Hava kalitesi ve kirlilik epizotları · **Veri:** UCI Beijing PM2.5 (PRSA, 2010–2014, saatlik)
**Girdi:** Proje 1'in temizlenmiş verisi, değerlendirme hattı (dokunulmayan 2014 test dönemi, `TimeSeriesSplit(5, gap=24)`) ve en iyi modeli (rastgele orman).

## 0. Hazırlık

In [ ]:
import sys
import warnings
import numpy as np
import pandas as pd
import sklearn
import scipy
import matplotlib
import matplotlib.pyplot as plt

from scipy.stats import spearmanr
from sklearn.base import clone
from sklearn.model_selection import TimeSeriesSplit, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.neighbors import LocalOutlierFactor, NearestNeighbors
from sklearn.cluster import KMeans, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.metrics import f1_score, balanced_accuracy_score, adjusted_rand_score, silhouette_score

print("=== KÜTÜPHANE SÜRÜMLERİ ===")
print(f"Python       : {sys.version.split()[0]}")
print(f"Pandas       : {pd.__version__}")
print(f"NumPy        : {np.__version__}")
print(f"Scikit-learn : {sklearn.__version__}")
print(f"SciPy        : {scipy.__version__}")
print(f"Matplotlib   : {matplotlib.__version__}")

SEED = 42
np.random.seed(SEED)
rng = np.random.default_rng(SEED)
print("\nRastgelelik tohumu (SEED) 42 olarak sabitlendi.")

### Okuma notları
- **Ben-Hur, Elisseeff ve Guyon (2002), A stability based method for discovering structure in clustered data:** _(2–3 cümle: doğru k'da alt örneklemlerin kümelemeleri birbirine benzer; kararlılık dağılımı k seçiminde nasıl kullanılır)_
- **Liu, Ting ve Zhou (2008), Isolation forest:** _(2–3 cümle: aykırılar az ve farklı olduğu için rastgele bölmelerle daha erken yalıtılır; yol uzunluğu skoru)_
- **Mehrabi vd. (2021), A survey on bias and fairness in machine learning:** _(2–3 cümle: yanlılık kaynakları; kişi bilgisi olmayan veride alt grup analizi)_

## 1. Proje 1'den devralınanlar ve düzeltmeler
_(Proje 1 geri bildirimi geldikten sonra doldurulacak: hangi geri bildirim geldi, ne düzeltildi, sonuçlar nasıl değişti. Proje 1'de planlananlar: `year` ve `day` sütunlarının çıkarılması, gradyan artırmanın ikinci aday olarak incelenmesi.)_

### 1.1 Veri ve değerlendirme hattı (Proje 1 ile aynı)

In [ ]:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00381/PRSA_data_2010.1.1-2014.12.31.csv"
df = pd.read_csv(url)

df_clean = df.drop(columns=['No']).copy()
df_clean['datetime'] = pd.to_datetime(df_clean[['year', 'month', 'day', 'hour']])
df_clean = df_clean.sort_values('datetime').dropna(subset=['pm2.5']).reset_index(drop=True)
df_clean['kirlilik_epizodu'] = (df_clean['pm2.5'] > 75).astype(int)

X = df_clean.drop(columns=['datetime', 'pm2.5', 'kirlilik_epizodu'])
y = df_clean['kirlilik_epizodu']
cat_cols = ['cbwd']
num_cols = [c for c in X.columns if c not in cat_cols]

# Dokunulmayan zamansal test: son %20 (2014). Bütün analizler geliştirme verisinde.
split_idx = int(len(X) * 0.8)
X_dev, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_dev, y_test = y.iloc[:split_idx], y.iloc[split_idx:]
dev = df_clean.iloc[:split_idx].copy()
tscv = TimeSeriesSplit(n_splits=5, gap=24)

# Proje 1'den aktarılan model: rastgele orman, aynı ön işleme hattıyla
preprocessor = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_cols),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')),
                      ('onehot', OneHotEncoder(handle_unknown='ignore'))]), cat_cols)
])
model_p1 = Pipeline([('preprocessor', preprocessor),
                     ('classifier', RandomForestClassifier(n_estimators=100, max_depth=10,
                                                           random_state=SEED, n_jobs=-1))])

print(f"Satır: {len(df_clean)} · geliştirme: {len(X_dev)} ({dev['datetime'].iloc[0]:%Y-%m-%d} → {dev['datetime'].iloc[-1]:%Y-%m-%d}) · test: {len(X_test)}")
print(f"Epizot oranı · geliştirme: %{y_dev.mean()*100:.1f} · test: %{y_test.mean()*100:.1f}")

## Bölüm 1 · Kümeleme
**Soru:** Hava koşulları birkaç belirgin "hava tipine" ayrılıyor mu, ve bu tipler kirlilik epizotlarıyla ilişkili mi?

**Kurulum:** Kümeleme yalnızca **dört hava değişkeniyle** yapılıyor: çiy noktası, sıcaklık, basınç ve birikimli rüzgâr hızı. Kar ve yağmur saatleri (`Is`, `Ir`) saatlerin %96–99'unda sıfır olduğu için kümelemeye alınmadı: neredeyse sabit sütunlar uzaklığı ve yoğunluğu "yağış var / yok" ayrımına indirgeyip yapay kümeler üretir. Bu iki sütun yalnızca profillerde gösteriliyor. Hedef (`pm2.5`, epizot) kümelemeye girmiyor; yalnızca kümeler bulunduktan sonra profilleri yorumlamak için kullanılıyor. Zaman sütunları (`year`, `month`, `day`, `hour`) da girmiyor; mevsimsellik kümelerden sonra profil olarak inceleniyor. Birikimli rüzgâr (`Iws`) çok çarpık olduğu için `log1p` dönüşümünden geçiyor. Test dönemi (2014) burada da kullanılmıyor.

### 1.1 Kümeleme eğilimi (Hopkins istatistiği) ve ölçeklemenin etkisi

In [ ]:
hava_cols = ['DEWP', 'TEMP', 'PRES', 'Iws', 'Is', 'Ir']    # profiller ve aykırılık için
kume_cols = ['DEWP', 'TEMP', 'PRES', 'Iws']                # kümeleme uzayı
log_cols = ['Iws', 'Is', 'Ir']

def log_donustur(A):
    A = np.asarray(A, dtype=float).copy()
    for j, c in enumerate(kume_cols):
        if c in log_cols:
            A[:, j] = np.log1p(A[:, j])
    return A

olcekli_hat = Pipeline([('log', FunctionTransformer(log_donustur)), ('scaler', StandardScaler())])
olceksiz_hat = Pipeline([('log', FunctionTransformer(log_donustur))])

# Hopkins, DBSCAN ve kararlılık analizi O(n²)'ye yakın; geliştirme verisinden rastgele 6.000 saatlik örneklem
orneklem_idx = np.sort(rng.choice(len(dev), size=min(6000, len(dev)), replace=False))
H_ham = dev[kume_cols].values[orneklem_idx]
Z = olcekli_hat.fit_transform(H_ham)          # kümeleme uzayı (ölçekli)
Z_olceksiz = olceksiz_hat.fit_transform(H_ham)

def hopkins(A, m_oran=0.05, tekrar=10, seed=SEED):
    """H ≈ 0,5: rastgele (yapı yok) · H → 1: kümelenme eğilimi var."""
    r = np.random.default_rng(seed)
    n, d = A.shape
    m = max(50, int(m_oran * n))
    nn = NearestNeighbors(n_neighbors=2).fit(A)
    lo, hi = A.min(axis=0), A.max(axis=0)
    sonuc = []
    for _ in range(tekrar):
        U = r.uniform(lo, hi, size=(m, d))
        u = nn.kneighbors(U, n_neighbors=1)[0][:, 0]
        ornek = A[r.choice(n, size=m, replace=False)]
        w = nn.kneighbors(ornek, n_neighbors=2)[0][:, 1]
        sonuc.append(u.sum() / (u.sum() + w.sum()))
    return np.mean(sonuc), np.std(sonuc)

def sutun_karistir(A, seed=SEED):
    """Her sütunu ayrı karıştırır: dağılımlar aynı kalır, sütunlar arası ilişki kaybolur."""
    r = np.random.default_rng(seed)
    return np.column_stack([r.permutation(A[:, j]) for j in range(A.shape[1])])

hop_sonuc = {'Ölçekli (StandardScaler)': hopkins(Z),
             'Ölçeksiz (ham birimler)': hopkins(Z_olceksiz),
             'Ölçekli · sütunlar karıştırılmış (yapısız karşılaştırma)': hopkins(sutun_karistir(Z))}
hop = pd.DataFrame(hop_sonuc, index=['Hopkins H (ortalama)', 'std (10 tekrar)']).T.round(3)
display(hop)

print("Ölçeksiz veride sütunların std'si (birimler farklı, büyük olan uzaklığı belirler):")
display(pd.Series(Z_olceksiz.std(axis=0), index=[f'{c} (log)' if c in log_cols else c for c in kume_cols]).round(2).to_frame('std'))

# Ölçeklemenin kümelemeye etkisi: aynı k ile ölçekli ve ölçeksiz KMeans ne kadar benzer?
km_o = KMeans(n_clusters=4, n_init=10, random_state=SEED).fit_predict(Z)
km_u = KMeans(n_clusters=4, n_init=10, random_state=SEED).fit_predict(Z_olceksiz)
print(f"k=4 için ölçekli ve ölçeksiz KMeans uyumu (ARI): {adjusted_rand_score(km_o, km_u):.3f}")

**Yorum:** _(Ölçekli veride H 0,5'ten ne kadar büyük? Sütunlar karıştırılınca H düşüyor mu? Düşüyorsa yapı sütunlar arası ilişkiden geliyor demektir. Ölçeksiz veride hangi sütun uzaklığa hâkim? ARI düşükse ölçekleme sonucu değiştiriyor demektir.)_

### 1.2 k seçimi: dirsek yöntemi ve kararlılık analizi (Ben-Hur vd., 2002)

In [ ]:
k_aralik = range(2, 11)
B = 20                       # kararlılık için alt örneklem çifti sayısı
oran = 0.8

ataletler, kar_ort, kar_std = [], [], []
for k in k_aralik:
    ataletler.append(KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(Z).inertia_)
    ari = []
    for b in range(B):
        r = np.random.default_rng(SEED + 1000 * k + b)
        i1 = r.choice(len(Z), size=int(oran * len(Z)), replace=False)
        i2 = r.choice(len(Z), size=int(oran * len(Z)), replace=False)
        ortak = np.intersect1d(i1, i2)
        m1 = KMeans(n_clusters=k, n_init=3, random_state=b).fit(Z[i1])
        m2 = KMeans(n_clusters=k, n_init=3, random_state=b + 500).fit(Z[i2])
        ari.append(adjusted_rand_score(m1.predict(Z[ortak]), m2.predict(Z[ortak])))
    kar_ort.append(np.mean(ari)); kar_std.append(np.std(ari))

k_tablo = pd.DataFrame({'atalet (inertia)': ataletler, 'kararlılık ARI ort.': kar_ort,
                        'kararlılık ARI std': kar_std}, index=pd.Index(list(k_aralik), name='k')).round(3)
display(k_tablo)

fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
ax[0].plot(list(k_aralik), ataletler, 'o-', color='#2F6E96')
ax[0].set_xlabel('Küme sayısı k'); ax[0].set_ylabel('Atalet (küme içi kareler toplamı)')
ax[0].set_title('Dirsek yöntemi'); ax[0].grid(True, ls='--', alpha=0.5)
ax[1].errorbar(list(k_aralik), kar_ort, yerr=kar_std, fmt='o-', capsize=5, color='#C4661F')
ax[1].set_xlabel('Küme sayısı k'); ax[1].set_ylabel('Alt örneklem çiftleri arası ARI (0–1)')
ax[1].set_title(f'Kararlılık ({B} çift, %{int(oran*100)} alt örneklem)'); ax[1].grid(True, ls='--', alpha=0.5)
fig.text(0.5, -0.04, 'Sol: atalet k arttıkça hep düşer, kırılma noktası aranır. Sağ: aynı k ile iki alt örneklemin '
         'kümelemeleri ortak noktalarda ne kadar uyuşuyor; 1 = tamamen kararlı.', ha='center', fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
# k seçimi kuralı: k = 2 neredeyse her veride "iki yarıya bölme" olarak kararlıdır ve bu veride sıcak / soğuk
# mevsim ayrımına karşılık gelir; bu yüzden k ≥ 3 arasından, hem ortalaması yüksek hem dağılımı dar olan
# (ortalama − std en büyük) k seçiliyor.
aday = k_tablo.loc[k_tablo.index >= 3]
puan = aday['kararlılık ARI ort.'] - aday['kararlılık ARI std']
K = int(puan.idxmax())
print("Kararlılık puanı (ortalama − std), k ≥ 3:", puan.round(3).to_dict())
print(f"Seçilen k = {K}")

**Yorum:** _(Dirsek nerede kırılıyor, net mi? Kararlılık hangi k'larda yüksek? İki yöntem aynı k'yı mı gösteriyor? k = 2 neden seçilmedi (önemsiz biçimde kararlı)? Ortalama − std kuralı neden makul? Ben-Hur vd.'nin kararlılık dağılımı önerisine dayandırın.)_

### 1.3 Yöntemlerin karşılaştırması: KMeans, Gauss karışımı ve DBSCAN (yoğunluk temelli)

In [ ]:
# DBSCAN için eps: min_samples'ıncı komşuya uzaklık eğrisinin (k-uzaklık grafiği) dirseği
min_samples = 2 * Z.shape[1]
k_uzak = np.sort(NearestNeighbors(n_neighbors=min_samples).fit(Z).kneighbors(Z)[0][:, -1])
eps = float(np.percentile(k_uzak, 90))

etiketler = {
    f'KMeans (k={K})': KMeans(n_clusters=K, n_init=10, random_state=SEED).fit_predict(Z),
    f'Gauss karışımı (k={K})': GaussianMixture(n_components=K, n_init=3, random_state=SEED).fit(Z).predict(Z),
    f'DBSCAN (eps={eps:.2f}, min_samples={min_samples})': DBSCAN(eps=eps, min_samples=min_samples).fit_predict(Z),
}

satir = []
for ad, e in etiketler.items():
    gurultu = (e == -1)
    kume_say = len(set(e[~gurultu]))
    sil = silhouette_score(Z[~gurultu], e[~gurultu]) if kume_say > 1 else np.nan
    satir.append({'Yöntem': ad, 'küme sayısı': kume_say, 'gürültü %': round(gurultu.mean() * 100, 1),
                  'silhouette (gürültü hariç)': round(sil, 3)})
display(pd.DataFrame(satir))

adlar = list(etiketler)
uyum = pd.DataFrame(index=adlar, columns=adlar, dtype=float)
for a in adlar:
    for b in adlar:
        ortak = (etiketler[a] != -1) & (etiketler[b] != -1)
        uyum.loc[a, b] = adjusted_rand_score(etiketler[a][ortak], etiketler[b][ortak])
print("Yöntemler arası uyum (ARI, gürültü noktaları hariç):")
display(uyum.round(3))

# DBSCAN eps'e duyarlıdır: farklı eps değerlerinde kaç küme ve ne kadar gürültü çıkıyor?
tarama = []
for yuzdelik in (50, 70, 80, 90, 95):
    e_ = float(np.percentile(k_uzak, yuzdelik))
    lab = DBSCAN(eps=e_, min_samples=min_samples).fit_predict(Z)
    boy = pd.Series(lab[lab != -1]).value_counts()
    tarama.append({'eps yüzdeliği': yuzdelik, 'eps': round(e_, 3), 'küme sayısı': int(boy.size),
                   'gürültü %': round((lab == -1).mean() * 100, 1),
                   'en büyük küme %': round(boy.max() / len(lab) * 100, 1) if boy.size else 0.0})
print("DBSCAN eps taraması:")
display(pd.DataFrame(tarama))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(k_uzak, color='#2F6E96')
ax.axhline(eps, color='#C4661F', ls='--', label=f'Seçilen eps = {eps:.2f} (90. yüzdelik)')
ax.set_xlabel('Noktalar (uzaklığa göre sıralı)'); ax.set_ylabel(f'{min_samples}. komşuya uzaklık (ölçekli)')
ax.set_title('DBSCAN için k-uzaklık grafiği'); ax.legend(); ax.grid(True, ls='--', alpha=0.5)
fig.text(0.5, -0.04, 'Eğrinin dikleştiği nokta yoğun bölgelerle seyrek bölgeleri ayırır; eps bu dirseğin yakınından seçildi.',
         ha='center', fontsize=9)
plt.tight_layout(); plt.show()

**Yorum:** _(Üç yöntem aynı yapıyı mı buluyor? DBSCAN kaç küme ve ne kadar gürültü buldu; eps taramasında küçük eps çok sayıda küçük küme ve çok gürültü, büyük eps tek bir dev küme veriyorsa hava verisi yoğunluk boşluklarıyla ayrılmış kümeler değil, sürekli bir uzaydır. KMeans küresel kümeler varsayar, GMM eliptik, DBSCAN yoğunluk; bu veride hangisinin varsayımı daha uygun?)_

### 1.4 Küme profilleri ve "bu kümeler gerçek mi?"

In [ ]:
# KMeans örneklemde eğitildi; bütün geliştirme verisine atanıyor
km_final = Pipeline([('hat', clone(olcekli_hat)), ('km', KMeans(n_clusters=K, n_init=10, random_state=SEED))])
km_final.fit(H_ham)
dev['kume'] = km_final.predict(dev[kume_cols].values)

profil = dev.groupby('kume').agg(
    pay_yuzde=('kume', lambda s: len(s) / len(dev) * 100),
    DEWP=('DEWP', 'mean'), TEMP=('TEMP', 'mean'), PRES=('PRES', 'mean'),
    Iws_medyan=('Iws', 'median'), kar_saati=('Is', 'mean'), yagmur_saati=('Ir', 'mean'),
    epizot_orani=('kirlilik_epizodu', lambda s: s.mean() * 100),
    pm25_medyan=('pm2.5', 'median'))
profil['baskin_yon'] = dev.groupby('kume')['cbwd'].agg(lambda s: s.value_counts().index[0])
profil['kis_payi_%'] = dev.groupby('kume')['month'].agg(lambda s: s.isin([12, 1, 2]).mean() * 100)
profil['yaz_payi_%'] = dev.groupby('kume')['month'].agg(lambda s: s.isin([6, 7, 8]).mean() * 100)
display(profil.round(1))

# Profil ısı haritası: her değişkenin küme ortalaması, genel ortalamaya göre z-skoru
zp = (dev.groupby('kume')[hava_cols].mean() - dev[hava_cols].mean()) / dev[hava_cols].std()
fig, ax = plt.subplots(figsize=(8, 0.8 * K + 1.5))
im = ax.imshow(zp.values, cmap='RdBu_r', vmin=-1.5, vmax=1.5, aspect='auto')
ax.set_xticks(range(len(hava_cols))); ax.set_xticklabels(hava_cols)
ax.set_yticks(range(K)); ax.set_yticklabels([f'Küme {k} · epizot %{profil.loc[k, "epizot_orani"]:.0f}' for k in zp.index])
for i in range(K):
    for j in range(len(hava_cols)):
        ax.text(j, i, f'{zp.values[i, j]:.1f}', ha='center', va='center', fontsize=9)
ax.set_xlabel('Hava değişkeni'); ax.set_title('Küme profilleri (genel ortalamaya göre z-skoru)')
fig.colorbar(im, ax=ax, label='z-skoru')
fig.text(0.5, -0.03, 'Kırmızı: ortalamanın üstü, mavi: altı. Satır etiketinde kümedeki epizot oranı; hedef kümelemeye girmedi.',
         ha='center', fontsize=9)
plt.tight_layout(); plt.show()

# "Gerçek mi?" için ölçütler: kararlılık, yapısız veriyle karşılaştırma, epizot oranının kümelere göre ayrışması
Zk = sutun_karistir(Z)
sil_gercek = silhouette_score(Z, KMeans(n_clusters=K, n_init=10, random_state=SEED).fit_predict(Z))
sil_karisik = silhouette_score(Zk, KMeans(n_clusters=K, n_init=10, random_state=SEED).fit_predict(Zk))
print(f"Silhouette · gerçek veri: {sil_gercek:.3f} · sütunları karıştırılmış veri: {sil_karisik:.3f}")
print(f"Seçilen k={K} için kararlılık ARI: {k_tablo.loc[K, 'kararlılık ARI ort.']:.3f} ± {k_tablo.loc[K, 'kararlılık ARI std']:.3f}")
print(f"Epizot oranı kümeler arasında: en düşük %{profil['epizot_orani'].min():.0f}, en yüksek %{profil['epizot_orani'].max():.0f} "
      f"(genel %{dev['kirlilik_epizodu'].mean()*100:.0f})")

**Yorum:** _(Her kümeye fiziksel bir ad verin, örn. "kuru-soğuk kuzeybatı havası", "nemli-sıcak yaz havası", "durgun kış havası". "Gerçek mi?" savunması: (1) kararlılık yüksek mi, (2) sütunları karıştırılmış veride silhouette düşüyor mu, (3) üç yöntem benzer yapı mı buluyor, (4) kümeler mevsim ve rüzgâr yönüyle anlamlı biçimde örtüşüyor mu, (5) hedef kümelemeye girmediği hâlde epizot oranı kümelere göre ayrışıyor mu? Neyi söyleyemeyiz: k'nın "doğru" değeri yok; kümeler sürekli bir hava uzayının bölümlenmesi olabilir.)_

## Bölüm 2 · Aykırı gözlem
**Soru:** Hangi saatler olağandışı, bunlar veri hatası mı yoksa gerçek ama olağandışı kirlilik dönemleri mi, ve modeli nasıl etkiliyorlar?

**Kurulum:** Aykırılık skorları geliştirme verisinde, hava değişkenleri **ve PM2.5** ile hesaplanıyor; "olağandışı kirlilik dönemi" ancak PM2.5'i hava koşullarıyla birlikte görerek bulunabilir. Ölçekleme ve log dönüşümü 1.1'deki gibi.

### 2.1 İki yöntemle aykırılık skoru: Isolation Forest ve LOF

In [ ]:
ay_cols = ['pm2.5'] + hava_cols
def log_donustur_ay(A):
    A = np.asarray(A, dtype=float).copy()
    for j, c in enumerate(ay_cols):
        if c in log_cols or c == 'pm2.5':
            A[:, j] = np.log1p(A[:, j])
    return A
ay_hat = Pipeline([('log', FunctionTransformer(log_donustur_ay)), ('scaler', StandardScaler())])
A = ay_hat.fit_transform(dev[ay_cols].values)

iforest = IsolationForest(n_estimators=300, contamination='auto', random_state=SEED, n_jobs=-1).fit(A)
dev['skor_if'] = -iforest.score_samples(A)                       # büyük = daha aykırı
lof = LocalOutlierFactor(n_neighbors=35, n_jobs=-1).fit(A)
dev['skor_lof'] = -lof.negative_outlier_factor_                  # büyük = daha aykırı

dev['sira_if'] = dev['skor_if'].rank(ascending=False).astype(int)
dev['sira_lof'] = dev['skor_lof'].rank(ascending=False).astype(int)

rho, p = spearmanr(dev['skor_if'], dev['skor_lof'])
uyum_satir = []
for yuzde in (1, 2, 5):
    n_ust = int(len(dev) * yuzde / 100)
    ust_if = set(dev.nsmallest(n_ust, 'sira_if').index)
    ust_lof = set(dev.nsmallest(n_ust, 'sira_lof').index)
    uyum_satir.append({'en aykırı dilim': f'%{yuzde} ({n_ust} saat)',
                       'ortak saat': len(ust_if & ust_lof),
                       'Jaccard': round(len(ust_if & ust_lof) / len(ust_if | ust_lof), 3)})
print(f"İki skorun sıra korelasyonu (Spearman ρ): {rho:.3f}")
display(pd.DataFrame(uyum_satir))

fig, ax = plt.subplots(figsize=(6.5, 5))
ax.scatter(dev['sira_if'], dev['sira_lof'], s=2, alpha=0.25, color='#2F6E96')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('Isolation Forest sırası (1 = en aykırı, log)'); ax.set_ylabel('LOF sırası (1 = en aykırı, log)')
ax.set_title('İki yöntemin aykırılık sıralaması'); ax.grid(True, ls='--', alpha=0.5)
fig.text(0.5, -0.03, 'Köşegene yakın noktalar iki yöntemin aynı saati benzer sırada bulduğunu gösterir.',
         ha='center', fontsize=9)
plt.tight_layout(); plt.show()

**Yorum:** _(Sıralamalar ne kadar uyumlu? İki yöntem aykırılığı farklı tanımlıyor: Isolation Forest küresel olarak "az ve farklı" olanı (Liu vd., 2008), LOF ise komşularına göre seyrek olanı yakalar. Uyum düşükse bu bir hata değil, iki farklı aykırılık türüdür.)_

### 2.2 En aykırı 20 kayıt: veri hatası mı, gerçek ama olağandışı mı?

In [ ]:
# Bahar Bayramı (Çin Yeni Yılı) ilk günleri: havai fişekler kısa süreli PM2.5 sıçraması yaratır
bahar_bayrami = pd.to_datetime(['2010-02-14', '2011-02-03', '2012-01-23', '2013-02-10', '2014-01-31'])
def bayrama_yakin(t):
    return bool(np.any(np.abs((t.normalize() - bahar_bayrami).days) <= 1))

ilk20 = dev.assign(ort_sira=(dev['sira_if'] + dev['sira_lof']) / 2).nsmallest(20, 'ort_sira')
isaret = []
for _, r in ilk20.iterrows():
    n = []
    if r['DEWP'] > r['TEMP']: n.append('çiy noktası > sıcaklık (fiziksel olarak olanaksız)')
    if r['pm2.5'] >= 500: n.append('PM2.5 ≥ 500')
    if r['pm2.5'] <= 5: n.append('PM2.5 ≤ 5')
    if r['Iws'] >= 200: n.append('çok uzun süren rüzgâr')
    if r['Is'] >= 10 or r['Ir'] >= 10: n.append('uzun kar/yağmur')
    if bayrama_yakin(r['datetime']): n.append('Bahar Bayramı ±1 gün')
    isaret.append('; '.join(n))
ilk20 = ilk20.assign(isaretler=isaret)
display(ilk20[['datetime', 'pm2.5', 'DEWP', 'TEMP', 'PRES', 'cbwd', 'Iws', 'Is', 'Ir',
               'sira_if', 'sira_lof', 'isaretler']].reset_index(drop=True))

print("Kontrol · bütün geliştirme verisinde çiy noktası > sıcaklık olan saat:", int((dev['DEWP'] > dev['TEMP']).sum()))
print("Kontrol · ilk 20'nin aylara dağılımı:", ilk20['datetime'].dt.month.value_counts().sort_index().to_dict())

**Yorum:** _(Her kayıt için karar: veri hatası mı (fiziksel olarak olanaksız bir değer, ani tek saatlik sıçrama), gerçek ama olağandışı mı (örn. Ocak 2013'teki uzun epizot, Bahar Bayramı havai fişekleri, uzun kar/yağmur)? Kararlarınızı tabloya bir sütun olarak ekleyin ve gerekçesini yazın. Neyi söyleyemeyiz: kesin "hata" kararı için ham ölçüm kayıtları gerekir.)_

### 2.3 Aykırı kayıtlar çıkarıldığında ya da işaretlendiğinde Proje 1 modelinin değişimi

In [ ]:
# Sızıntısız kurulum: her zamansal katta aykırılık modeli YALNIZCA o katın eğitim verisiyle eğitilir.
#  - Çıkar: eğitim katındaki en aykırı %1 ve %2 saat çıkarılır; doğrulama katına dokunulmaz.
#  - İşaretle: aykırılık skoru öznitelik olarak eklenir. PM2.5 tahmin anında bilinmediği için bu skor
#    yalnızca girdi özniteliklerinden (hava + zaman) hesaplanır.
def if_skoru(tr_matris, uygula_matris):
    sc = StandardScaler().fit(tr_matris)
    m = IsolationForest(n_estimators=200, random_state=SEED, n_jobs=-1).fit(sc.transform(tr_matris))
    return -m.score_samples(sc.transform(uygula_matris))

Xd = X_dev.reset_index(drop=True); yd = y_dev.reset_index(drop=True)
dv = dev.reset_index(drop=True)
sonuc = {k: {'f1': [], 'bacc': []} for k in ['Proje 1 modeli (değişiklik yok)', 'Eğitimden en aykırı %1 çıkarıldı',
                                               'Eğitimden en aykırı %2 çıkarıldı', 'Aykırılık skoru öznitelik olarak eklendi']}
for tr, va in tscv.split(Xd):
    s_tr = if_skoru(log_donustur_ay(dv.loc[tr, ay_cols].values), log_donustur_ay(dv.loc[tr, ay_cols].values))
    for ad, oran_cikar in [('Proje 1 modeli (değişiklik yok)', 0), ('Eğitimden en aykırı %1 çıkarıldı', 1),
                           ('Eğitimden en aykırı %2 çıkarıldı', 2)]:
        tutulan = tr if oran_cikar == 0 else tr[s_tr < np.percentile(s_tr, 100 - oran_cikar)]
        m = clone(model_p1).fit(Xd.loc[tutulan], yd.loc[tutulan])
        t = m.predict(Xd.loc[va])
        sonuc[ad]['f1'].append(f1_score(yd.loc[va], t)); sonuc[ad]['bacc'].append(balanced_accuracy_score(yd.loc[va], t))
    # İşaretle: skor girdi özniteliklerinden (num_cols, cbwd hariç)
    g_tr, g_va = Xd.loc[tr, num_cols].values, Xd.loc[va, num_cols].values
    Xtr2 = Xd.loc[tr].assign(aykirilik=if_skoru(g_tr, g_tr))
    Xva2 = Xd.loc[va].assign(aykirilik=if_skoru(g_tr, g_va))
    pre2 = ColumnTransformer([('num', StandardScaler(), num_cols + ['aykirilik']),
                              ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)])
    m2 = Pipeline([('preprocessor', pre2), ('classifier', clone(model_p1.named_steps['classifier']))]).fit(Xtr2, yd.loc[tr])
    t2 = m2.predict(Xva2)
    ad = 'Aykırılık skoru öznitelik olarak eklendi'
    sonuc[ad]['f1'].append(f1_score(yd.loc[va], t2)); sonuc[ad]['bacc'].append(balanced_accuracy_score(yd.loc[va], t2))

etki = pd.DataFrame({ad: {'F1 (ortalama ± std)': f"{np.mean(v['f1']):.4f} ± {np.std(v['f1']):.4f}",
                          'Dengeli doğruluk (ortalama ± std)': f"{np.mean(v['bacc']):.4f} ± {np.std(v['bacc']):.4f}",
                          'F1 farkı (kat bazında ort.)': f"{np.mean(np.array(v['f1']) - np.array(sonuc['Proje 1 modeli (değişiklik yok)']['f1'])):+.4f}"}
                     for ad, v in sonuc.items()}).T
display(etki)

**Yorum:** _(Aykırıları çıkarmak modeli iyileştiriyor mu, kötüleştiriyor mu? Fark katlar arası std'nin yanında anlamlı mı? Aykırıların çoğu gerçek kirlilik dönemleriyse (2.2), onları çıkarmak modeli tam da önemli durumlardan mahrum bırakır. Skorun öznitelik olarak eklenmesi neden yalnızca girdilerle yapıldı (tahmin anında PM2.5 bilinmez)? Test dönemi kullanılmadı.)_

## Bölüm 3 · Alt grup analizi
_(Proje 1 geri bildiriminden sonra: modelin mevsime, günün saatine ve rüzgâr yönüne göre hata oranları; zayıf olduğu koşullar ve nedenleri. Mehrabi vd., 2021.)_

## Ek paket 1 · Kümeler ve sınıflandırma
_(Proje 1 geri bildiriminden sonra: modelin kümelere göre hata analizi; küme bilgisinin sızıntısız öznitelik olarak eklenmesi; kümelere göre ayrı modeller ile tek modelin karşılaştırılması.)_

## Tez ve araştırma yönleri
_(Yarım sayfa. Tez öncülü: "İstanbul gibi yerel bir şehrin açık hava kalitesi verisiyle kirlilik epizotlarını erken tespit eden uygulamalı bir tez." (1) Araştırma sorusu, (2) projeden çıkan gerekçe ve en az bir kaynak, (3) gereken veri, yöntem ve protokol, (4) fizibilite ve riskler.)_

## Model kartı
_(Amaç, veri, alt gruplar dahil başarım, sınırlılıklar, uygun olmayan kullanımlar.)_

## Sonuç ve sınırlılıklar

## Varsayım defteri

## Katkı beyanı

## Yapay zekâ beyanı

## Kaynaklar
- Ben-Hur, A., Elisseeff, A. ve Guyon, I. (2002). A stability based method for discovering structure in clustered data. *Pacific Symposium on Biocomputing*, 7, 6–17.
- Liu, F. T., Ting, K. M. ve Zhou, Z.-H. (2008). Isolation forest. *Proceedings of the IEEE International Conference on Data Mining (ICDM)*, 413–422.
- Mehrabi, N., Morstatter, F., Saxena, N., Lerman, K. ve Galstyan, A. (2021). A survey on bias and fairness in machine learning. *ACM Computing Surveys*, 54(6), 1–35.
- Liang, X. vd. (2015). Assessing Beijing's PM2.5 pollution: severity, weather impact, APEC and winter heating. *Proceedings of the Royal Society A*, 471, 20150257.